# 04 — CNN entraîné de zéro (PyTorch)

> **Projet AutoMaze AI** — classification de 6 usagers de la route, entraînée de zéro.
> Règles du cours : pas de YOLO, pas de détecteur, **pas de poids préentraînés** (`pretrained=True` interdit).
> Les cellules `TODO` sont à écrire par l'équipe. Ne supprimez pas les questions : répondez-y en dessous, elles nourrissent le rapport.

**Objectif :** concevoir, entraîner et régler un petit réseau convolutif qui bat la baseline HOG + SVM.

**Méthode (à suivre dans l'ordre) :**
1. vérifier les données chargées (une grille d'images avec leurs étiquettes) ;
2. vérifier que le modèle peut **sur-apprendre un mini-lot** (~50 images) : sinon, il y a un bug ;
3. entraîner le modèle de base, tracer les courbes train / validation ;
4. améliorer en changeant **une seule chose à la fois**, et noter chaque run ;
5. garder le meilleur modèle selon la **validation** ; le test reste fermé jusqu'au notebook 05.

**Interdit :** `torchvision.models.xxx(weights=...)`, `pretrained=True`, poids téléchargés depuis un hub.

**Sur Colab :** menu *Exécution > Modifier le type d'exécution > GPU (T4)*. La session gratuite peut être coupée
après quelques heures ou en cas d'inactivité : le meilleur modèle est donc sauvegardé dans Drive à chaque amélioration.

## 0. Imports

In [ ]:
import json
import time
from datetime import datetime

import matplotlib.pyplot as plt
import torch
import torch.nn as nn                      # couches (Conv2d, Linear, BatchNorm2d...)
import torch.nn.functional as F            # fonctions sans paramètres (relu, softmax...)
from torch.utils.data import DataLoader    # découpe les données en lots (batches)
from torchvision import datasets           # ImageFolder lit data/crops/<jeu>/<classe>/
from torchvision import transforms as T
from sklearn.metrics import f1_score

In [ ]:
# --- Configuration commune (identique dans chaque notebook) ---
from pathlib import Path
import random

import numpy as np

# Exécution sur Google Colab ou en local : le même notebook fonctionne dans les deux cas.
try:
    import google.colab  # type: ignore  # noqa: F401  (module présent uniquement sur Colab : avertissement Pylance normal en local)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Sur Colab, tout ce qui est écrit dans /content disparaît à la fin de la session.
    # On travaille donc dans Google Drive : données, figures et poids y sont conservés.
from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/AutoMaze_Ai")  # TODO(équipe) : adapter au dossier partagé de l'équipe
else:
    # En local, ce notebook est dans notebooks/ : la racine du projet est le dossier parent.
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Colab" if IN_COLAB else "Local", "| racine du projet :", ROOT)
DATA_DIR = ROOT / "data"          # données locales, ignorées par Git
RAW_DIR = DATA_DIR / "raw"        # fichiers téléchargés tels quels (ex. annotations COCO)
CROPS_DIR = DATA_DIR / "crops"    # recadrages produits par le notebook 02
MODELS_DIR = ROOT / "models"      # poids + métadonnées (les .pt sont ignorés par Git)
FIGURES_DIR = ROOT / "reports" / "figures"  # figures à reprendre dans le rapport

for d in (RAW_DIR, CROPS_DIR, MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ordre des classes = source de vérité pour TOUT le projet (notebooks + simulateur).
# Ne le changez plus une fois l'entraînement commencé : l'indice 0 doit toujours désigner la même classe.
CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]
CLASSES_FR = ["Personne", "Vélo", "Voiture", "Moto", "Bus", "Camion"]

# Graine aléatoire : rend les résultats reproductibles (même découpage, mêmes initialisations).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
torch.manual_seed(SEED)

# Utilise le GPU s'il existe (CUDA ou Apple MPS), sinon le CPU. Un petit CNN s'entraîne aussi sur CPU.
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Appareil utilisé :", DEVICE, "| PyTorch", torch.__version__)

## 1. Paramètres repris du notebook 02

Copiez les **mêmes** valeurs. Une différence ici = un modèle qui se comporte autrement dans le simulateur.

In [ ]:
INPUT_SIZE = None   # TODO(équipe)
MEAN = None         # TODO(équipe) : liste de 3 valeurs
STD = None          # TODO(équipe) : liste de 3 valeurs

train_transform = None  # TODO(équipe) : copie exacte du notebook 02
eval_transform = None   # TODO(équipe) : copie exacte du notebook 02

## 1 bis. Sur Colab : copier les images sur le disque local

Lire des milliers de petits fichiers directement depuis Drive est très lent et peut ralentir l'entraînement d'un facteur 10.
Conseil : le notebook 02 range `data/crops/` dans une archive `crops.zip` sur Drive ; ici, on la décompresse dans `/content` (rapide).

In [ ]:
TRAIN_DATA_DIR = CROPS_DIR
if IN_COLAB:
    # TODO(équipe) : décompresser DATA_DIR / "crops.zip" vers Path("/content/crops") si ce n'est pas déjà fait,
    # puis TRAIN_DATA_DIR = Path("/content/crops"). Les poids, eux, restent sauvegardés dans MODELS_DIR (Drive).
    pass

## 2. Datasets et DataLoaders

**Attention :** `ImageFolder` trie les dossiers par ordre alphabétique. Vérifiez que `class_to_idx` correspond à `CLASSES`, sinon remappez.

In [ ]:
# TODO(équipe) : train_ds / val_ds avec datasets.ImageFolder(TRAIN_DATA_DIR / "train", ...), puis DataLoader (batch_size à choisir, shuffle=True pour le train).
# TODO(équipe) : afficher train_ds.class_to_idx et comparer avec CLASSES.
# TODO(équipe) : afficher une grille d'un lot avec les noms de classes (dénormaliser pour l'affichage).

## 3. Architecture

Point de départ conseillé : 3 blocs `Conv2d → BatchNorm2d → ReLU → MaxPool2d`, en doublant le nombre de filtres
à chaque bloc (ex. 32, 64, 128), puis une couche `Linear` vers 6 sorties.

**Questions :** combien de paramètres a votre modèle ? Quelle taille a la carte de caractéristiques avant la couche dense ?
Pourquoi ne met-on pas de softmax dans `forward` quand on utilise `nn.CrossEntropyLoss` ?

In [ ]:
class AutoMazeCNN(nn.Module):
    """Petit CNN initialisé aléatoirement (aucun poids préentraîné)."""

    def __init__(self, num_classes=len(CLASSES)):
        super().__init__()
        # TODO(équipe) : définir les couches.

    def forward(self, x):
        # TODO(équipe) : enchaîner les couches et renvoyer les logits (pas de softmax).
        raise NotImplementedError


# model = AutoMazeCNN().to(DEVICE)
# print(model)
# print("Paramètres entraînables :", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Boucles d'entraînement et d'évaluation

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    """Une passe sur le train. Renvoie (perte moyenne, accuracy)."""
    model.train()   # active dropout / BatchNorm en mode entraînement
    # TODO(équipe) : pour chaque lot -> forward, perte, zero_grad, backward, step ; cumuler perte et bonnes réponses.
    raise NotImplementedError


@torch.no_grad()    # pas de calcul de gradient en évaluation : plus rapide, moins de mémoire
def evaluate(model, loader, criterion):
    """Renvoie (perte moyenne, accuracy, F1 macro, y_true, y_pred)."""
    model.eval()    # désactive dropout, BatchNorm utilise ses statistiques apprises
    # TODO(équipe)
    raise NotImplementedError

## 5. Test de sur-apprentissage d'un mini-lot

Entraînez ~100 itérations sur un seul lot de ~50 images. La perte doit tendre vers 0. Si ce n'est pas le cas, cherchez le bug avant d'aller plus loin.

In [ ]:
# TODO(équipe)

## 6. Entraînement complet

Hyperparamètres de départ raisonnables : optimiseur Adam, taux d'apprentissage 1e-3, 20 à 30 époques.
Sauvegardez le modèle **chaque fois que le F1 macro de validation s'améliore** (early stopping).

In [ ]:
RUN_NAME = "run02_cnn_base"   # un nom par essai, repris dans le journal des runs du rapport
EPOCHS = None                 # TODO(équipe)
LR = None                     # TODO(équipe)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

# TODO(équipe) : optimizer, criterion (penser à weight= pour le déséquilibre ?), boucle sur les époques,
# remplissage de history, sauvegarde du meilleur state_dict dans MODELS_DIR / f"{RUN_NAME}.pt".

## 7. Courbes d'apprentissage

**Lecture :** perte train ↓ et perte val ↑ = sur-apprentissage. Les deux hautes = sous-apprentissage.

In [ ]:
# TODO(équipe) : deux graphiques côte à côte (pertes, puis accuracy / F1), sauvegardés dans FIGURES_DIR.

## 8. Pistes d'amélioration (une à la fois)

| Problème observé | Pistes |
|---|---|
| Sur-apprentissage | plus d'augmentation, Dropout, weight decay, modèle plus petit |
| Sous-apprentissage | plus de filtres / de blocs, plus d'époques, taux d'apprentissage ajusté |
| Classe rare mal reconnue | poids de classe dans la perte, plafond plus bas sur les classes fréquentes |
| Courbes instables | taux d'apprentissage plus faible, scheduler (`StepLR`, `ReduceLROnPlateau`) |

## 9. Export pour le simulateur

Le simulateur chargera **uniquement** ces fichiers ; il ne relance jamais l'entraînement.
Quand l'architecture est figée, copiez la classe `AutoMazeCNN` dans `src/` (Claude Code s'en chargera avec vous)
pour que le simulateur puisse reconstruire le modèle.

In [ ]:
# Contrat attendu par le simulateur : ne pas renommer les clés.
metadata = {
    "model_name": RUN_NAME,
    "weights_path": f"models/{RUN_NAME}.pt",       # state_dict du meilleur modèle
    "architecture": "AutoMazeCNN",
    "classes": CLASSES,                            # ordre = indices de sortie du modèle
    "classes_fr": CLASSES_FR,
    "input_size": [INPUT_SIZE, INPUT_SIZE],
    "color_order": "RGB",
    "mean": MEAN,
    "std": STD,
    "framework": f"pytorch {torch.__version__}",
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "val_f1_macro": None,                          # TODO(équipe) : valeur réelle mesurée, jamais inventée
}
# TODO(équipe) : une fois le meilleur run choisi, écrire ce dictionnaire dans MODELS_DIR / "metadata.json".